In [1]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

In [2]:
df_reporte_Emision= pd.read_csv("C:/data/T-1378171-002.csv", dtype=str)
df_reporte_Emision.columns = (df_reporte_Emision.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True))  

In [3]:
df_reporte_Emision.drop(['_DESGRAVAMENBBVAKT_','DSCMOVIMIENTO','ESTADO','_EMITIDOENA_X____LOT_IDEENTIDAD__FINN_132018_01'], axis=1, inplace=True)

In [4]:
df_reporte_Emision= df_reporte_Emision.rename(columns={'NUMCERTIFICADOCANAL':'CERTIFICADO', 'NOMARCHIVO':'NOMBRE_ARCHIVO', 
                                                       'FECINIVIGCAN':'FECHA_INICIO', 'FECFINVIGCAN':'FECHA_FIN', 
                                                       'PRIMABRUTAEMI':'PRIMA_BRUTA','DIV':'MONEDA'})

In [ ]:
df_reporte_Emision['CERTIFICADO'] = df_reporte_Emision['CERTIFICADO'].str.replace(r'\D', '', regex=True) # eliminar caracteres no numéricos
df_reporte_Emision['FECHA_INICIO'] = pd.to_datetime(df_reporte_Emision['FECHA_INICIO'],format='%d/%m/%Y', errors='coerce').dt.date
df_reporte_Emision['FECHA_FIN'] = pd.to_datetime(df_reporte_Emision['FECHA_FIN'],format='%d/%m/%Y', errors='coerce').dt.date
df_reporte_Emision['PRIMA_BRUTA'] = pd.to_numeric(df_reporte_Emision['PRIMA_BRUTA'], errors="coerce").astype('float64')

In [8]:
df_reporte_Emision.head(5)

,IDELOTE,IDEDET,CERTIFICADO,NOMBRE_ARCHIVO,FECHA_INICIO,FECHA_FIN,MONEDA,PRIMA_BRUTA,IDPPROCESO,NROCOBRO
0,129268,227721063,00110180784000132603,20100130204_0155001_20180403_002.TXT,2018-02-20,2018-03-20,SOL,0.47,REP,626357779
1,129268,227721951,00110237594000199650,20100130204_0155001_20180403_002.TXT,2018-02-20,2018-03-20,SOL,52.47,REP,626352442
2,129268,227720737,00110155904000138524,20100130204_0155001_20180403_002.TXT,2018-02-10,2018-03-09,SOL,87.21,REP,626355510
3,129268,227720983,00110175764000243904,20100130204_0155001_20180403_002.TXT,2018-02-20,2018-03-20,SOL,17.83,REP,626355865
4,129268,227721240,00110201114000141842,20100130204_0155001_20180403_002.TXT,2018-02-10,2018-03-09,SOL,12.28,REP,626352062


In [23]:
df_reporte_Emision.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147080 entries, 0 to 147079
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   IDELOTE         147080 non-null  object 
 1   IDEDET          147080 non-null  object 
 2   CERTIFICADO     147080 non-null  object 
 3   NOMBRE_ARCHIVO  147080 non-null  object 
 4   FECHA_INICIO    147080 non-null  object 
 5   FECHA_FIN       147080 non-null  object 
 6   MONEDA          147080 non-null  object 
 7   PRIMA_BRUTA     146450 non-null  float64
 8   IDPPROCESO      147080 non-null  object 
 9   NROCOBRO        147080 non-null  object 
dtypes: float64(1), object(9)
memory usage: 11.2+ MB


In [ ]:
schema_Reporte_Emision = [
        bigquery.SchemaField("IDELOTE", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("IDEDET", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("IDPPROCESO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NROCOBRO", bigquery.enums.SqlTypeNames.STRING),
]